# 01 - Preprocessing

Run after `00_setup.ipynb`.

Reads the raw MATLAB output `image/*.png` + `label/*.txt` under `$DATA_ROOT/processed/{duke_dme,hc_ms}`
(written by `generate_dme_train.m` / `generate_hc_train.m`):

1. applies `src/s2_preprocessing/` (denoise, normalization; flattening/cropping already done) and
   writes `image/*.npy` + `label/*.txt` to `$DATA_ROOT/processed/{duke_dme,hc_ms}_denoised`
2. writes patient-level splits over the denoised datasets.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

from src.s1_data.dataset import OCTDataset
from src.s1_data.splits import build_patient_groups, patient_folds, save_folds
from src.s2_preprocessing.denoise import denoise, estimate_sigma
from src.s2_preprocessing.normalize import normalize


def preprocess(bscan):
    return normalize(denoise(bscan, sigma=estimate_sigma(bscan)))


def preprocess_and_save(raw_dir, output_dir):
    """Read raw image/*.png + label/*.txt under raw_dir (MATLAB output),
    denoise+normalize each B-scan, and write image/*.npy + a copy of
    label/*.txt (boundaries are unaffected by denoising) to output_dir —
    the image/*.npy + label/*.txt layout OCTDataset expects.

    Reads raw_dir directly rather than through OCTDataset: OCTDataset only
    ever sees the denoised+normalized *.npy output, never the raw *.png.
    """
    raw_dir = Path(raw_dir)
    output_dir = Path(output_dir)
    image_dir = output_dir / "image"
    label_dir = output_dir / "label"
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    n = 0
    for image_path in sorted((raw_dir / "image").glob("*.png")):
        label_path = raw_dir / "label" / (image_path.stem + ".txt")
        if not label_path.exists():
            continue
        bscan = np.asarray(Image.open(image_path))
        np.save(image_dir / f"{image_path.stem}.npy", preprocess(bscan))
        shutil.copy(label_path, label_dir / label_path.name)
        n += 1

    print(f"OK   {n} B-scans -> {output_dir}")


# 1. Denoise + normalize, save as .npy -------------------------------------
preprocess_and_save(DATA_ROOT / 'processed/duke_dme', DATA_ROOT / 'processed/duke_dme_denoised')
preprocess_and_save(DATA_ROOT / 'processed/hc_ms', DATA_ROOT / 'processed/hc_ms_denoised')

# 2. Patient-level 5-fold CV split, stratified by group (DME/HC/MS) --------
duke_dme = OCTDataset(DATA_ROOT / 'processed/duke_dme_denoised')
hc_ms = OCTDataset(DATA_ROOT / 'processed/hc_ms_denoised')
patient_groups = build_patient_groups(duke_dme.patient_ids(), hc_ms.patient_ids())
folds = patient_folds(patient_groups)

folds_path = DATA_ROOT / 'processed/folds.json'
save_folds(folds, folds_path)
print(f"OK   {len(folds)} folds over {len(patient_groups)} patients "
      f"({len(duke_dme.patient_ids())} DME, {len(hc_ms.patient_ids())} HC-MS) -> {folds_path}")

# 02_run_methods.ipynb reads folds.json directly (src.s1_data.splits.load_folds)
# instead of recomputing — so it doesn't depend on rerunning this notebook.